# FinSight Phase 1 — Deep Dive (for a complete RAG beginner)

**Who this notebook is for:** you can code and you know ML (CNNs, embeddings,
training loops) but you have **never built a RAG system or touched a vector
database**. Nothing here is assumed. Every step shows the **real data** before
and after it is transformed.

> If you already know RAG and just want the fast tour, open
> `phase1_walkthrough.ipynb` instead. This notebook is the slow, spoon-fed one.

---

## First: what are we even building?

A user asks a plain-English question about a company's financial filings:

> *"What cybersecurity risks did JPMorgan flag in its latest annual report?"*

and gets back an answer where **every sentence has a citation** pointing to the
exact paragraph in the real document — or the system **refuses** if it doesn't
have the evidence. No made-up facts. That last property is the whole point:
in finance, a confident wrong answer is worse than "I don't know."

The technique that makes this possible is **RAG — Retrieval-Augmented
Generation**. Here it is in one sentence:

> Instead of asking a language model to answer from memory, we first **retrieve**
> the most relevant paragraphs from real documents, paste them into the prompt,
> and tell the model: *"answer using only this."*

### An analogy you already know

Think of the difference between a **closed-book exam** and an **open-book exam**:

- A plain chatbot (ChatGPT with no tools) is a student answering from memory —
  fluent, but it will confidently invent a number if it doesn't remember.
- **RAG is the open-book exam** — the student first finds the right page, then
  answers with the book open, and writes down the page number next to each claim.

Everything in Phase 1 is about building that "open book" and the machinery to
find the right page fast.

### Mapping RAG jargon to concepts you already have (CV background)

| You already know (CV / ML) | The RAG equivalent in this project |
|---|---|
| A CNN turns an image into a **feature vector** | An *embedding model* turns a **paragraph of text** into a vector |
| Two similar images → similar feature vectors | Two similar *meanings* → similar text vectors |
| **Cosine similarity** to compare feature vectors | Same cosine similarity, to compare text meaning |
| **FAISS** / a nearest-neighbour index | A **vector database** (here: Postgres + pgvector) |
| Tiling a big image into **patches** before processing | **Chunking** a 300-page document into passages |
| A labelled dataset you can evaluate against | A "golden dataset" of Q&A (that's Phase 2) |

Keep this table in mind — whenever a new term appears, it probably maps to
something you already understand.

---

## The Phase 1 pipeline, end to end

```
   SEC EDGAR website          ┌─────────────── OUR CODE ───────────────┐
   (public filings)           │                                        │
        │                     │  1. download   2. clean    3. split    │
        ▼                     │     HTML          text        sections │
   raw .htm file  ───────────►│                                        │
   (12.9 MB soup)             │  4. chunk    5. embed    6. store rows  │
                              │     + tags      vectors     in database │
                              └───────────────────┬────────────────────┘
                                                   ▼
        answer  ◄── 8. LLM writes ◄── 7. search the database ◄── your question
      (with citations)          cited answer   (nearest vectors)
```

We will walk **every one of these 8 steps** with the real data in front of us.
Let's set up first.

In [1]:
# --- Setup: make the repo's modules importable from the notebooks/ folder ---
import sys, json, textwrap
from pathlib import Path

# When you run a notebook, the "current folder" is wherever the notebook lives
# (notebooks/). Our code lives one level up, so we add the repo root to the path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)

from IPython.display import HTML, Markdown, display

# A tiny helper to show BEFORE vs AFTER side by side, so every transformation
# in this notebook is visible, not just described.
def before_after(before, after, before_title="BEFORE", after_title="AFTER"):
    def esc(t):
        return (str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
    display(HTML(f'''
    <div style="display:flex; gap:12px; flex-wrap:wrap">
      <div style="flex:1; min-width:280px; border:1px solid #c33; border-radius:8px; overflow:hidden">
        <div style="background:#c33; color:#fff; padding:4px 10px; font-weight:700">{before_title}</div>
        <pre style="margin:0; padding:10px; white-space:pre-wrap; font-size:12px; max-height:320px; overflow:auto">{esc(before)}</pre>
      </div>
      <div style="flex:1; min-width:280px; border:1px solid #2a8; border-radius:8px; overflow:hidden">
        <div style="background:#2a8; color:#fff; padding:4px 10px; font-weight:700">{after_title}</div>
        <pre style="margin:0; padding:10px; white-space:pre-wrap; font-size:12px; max-height:320px; overflow:auto">{esc(after)}</pre>
      </div>
    </div>'''))

print("repo root:", ROOT)
print("Python   :", sys.version.split()[0])

repo root: C:\Users\subha\OneDrive\Desktop\Subham\finsight
Python   : 3.14.6


# Step 1 — Where does the data come from?

**Short answer:** the U.S. government. Every public company (Apple, JPMorgan…)
is legally required to file detailed reports with the **SEC** (Securities and
Exchange Commission). Those filings are published, for free, on a system called
**EDGAR**. There is even a free API — no key, no signup.

The report we care about most is the **10-K**: a company's annual report. It's
huge (200–400 pages) and has a fixed structure with numbered sections called
"Items":

- **Item 1A — Risk Factors**: everything that could go wrong (this is gold)
- **Item 7 — MD&A**: management explaining the numbers in plain-ish English
- **Item 8 — Financial Statements**: the actual accounting tables

### Did we download HTML, or JSON, or both?

Great question to get straight early, because it confuses everyone:

- From the SEC we download **HTML** — the filing is a web page (a `.htm` file).
- The **JSON** files you saw (`JPM_10-K_2025-12-31.jsonl`) are **not** from the
  SEC. **We create them ourselves** later, after cleaning and chunking. They're
  our processed output, not the raw input.
- Separately, the SEC API *also* uses JSON for its **directory/index** endpoints
  (the "which filings exist" lookup) — but the filing *content* itself is HTML.

So the flow is: **SEC gives us HTML → our code produces JSON.** Let's see it.

### 1a — The SEC's index endpoints (these return JSON)

Before we can download JPMorgan's 10-K, we need to know *where it lives*. The
SEC identifies every company by a number called a **CIK** (Central Index Key),
not by its ticker symbol. So there are two lookups:

1. ticker `"JPM"` → CIK `19617`  (via a big lookup file)
2. CIK `19617` → list of all its filings  (via the "submissions" endpoint)

Let's call both and look at the **raw JSON** the SEC returns.

In [2]:
import httpx
from config import EDGAR_USER_AGENT

# The SEC REQUIRES a User-Agent header identifying who you are (an email).
# They block anonymous scrapers. This is set in your .env file.
http = httpx.Client(headers={"User-Agent": EDGAR_USER_AGENT}, timeout=30, follow_redirects=True)
print("Identifying ourselves to the SEC as:", EDGAR_USER_AGENT)

# --- Lookup 1: the ticker -> CIK map. This is one big JSON file. ---
ticker_map_raw = http.get("https://www.sec.gov/files/company_tickers.json").json()
print("\nThe raw file is a dict of row-number -> record. First 2 records:")
print(json.dumps(dict(list(ticker_map_raw.items())[:2]), indent=2))

Identifying ourselves to the SEC as: FinSight research project subham.tiwari186@gmail.com



The raw file is a dict of row-number -> record. First 2 records:
{
  "0": {
    "cik_str": 1045810,
    "ticker": "NVDA",
    "title": "NVIDIA CORP"
  },
  "1": {
    "cik_str": 320193,
    "ticker": "AAPL",
    "title": "Apple Inc."
  }
}


In [3]:
# Find JPMorgan's record
jpm = next(v for v in ticker_map_raw.values() if v["ticker"] == "JPM")
cik = jpm["cik_str"]
print("JPMorgan's SEC record:", jpm)
print("=> CIK (the SEC's ID for JPM):", cik)

JPMorgan's SEC record: {'cik_str': 19617, 'ticker': 'JPM', 'title': 'JPMORGAN CHASE & CO'}
=> CIK (the SEC's ID for JPM): 19617


In [4]:
# --- Lookup 2: all of JPM's filings. The CIK is zero-padded to 10 digits. ---
subs = http.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()
print("Company name :", subs["name"])
print("Ticker(s)    :", subs["tickers"])

# The list of recent filings is stored as PARALLEL ARRAYS (column-oriented):
# one array of form types, one of dates, etc. Entry i of each array = filing i.
recent = subs["filings"]["recent"]
print("\nKeys describing each filing:", list(recent.keys())[:8], "...")
print("Number of filings in this index window:", len(recent["form"]))

Company name : JPMORGAN CHASE & CO
Ticker(s)    : ['JPM', 'JPM-PC', 'AMJB', 'JPM-PD', 'JPM-PJ', 'JPM-PK', 'JPM-PL', 'JPM-PM', 'VYLD']

Keys describing each filing: ['accessionNumber', 'filingDate', 'reportDate', 'acceptanceDateTime', 'act', 'form', 'fileNumber', 'filmNumber'] ...
Number of filings in this index window: 25450


In [5]:
# Those parallel arrays are awkward. Let's zip them into a normal table and
# keep only the annual (10-K) and quarterly (10-Q) reports.
filings_df = pd.DataFrame({
    "form": recent["form"],
    "filingDate": recent["filingDate"],
    "reportDate": recent["reportDate"],
    "primaryDocument": recent["primaryDocument"],
    "accessionNumber": recent["accessionNumber"],
})
display(filings_df[filings_df.form.isin(["10-K", "10-Q"])].head(8))
print("^ 'primaryDocument' is the actual HTML file name we need to download.")

,form,filingDate,reportDate,primaryDocument,accessionNumber
5660,10-Q,2026-05-01,2026-03-31,jpm-20260331.htm,0001628280-26-029344
11484,10-K,2026-02-13,2025-12-31,jpm-20251231.htm,0001628280-26-008131
18515,10-Q,2025-11-04,2025-09-30,jpm-20250930.htm,0001628280-25-048859
24061,10-Q,2025-08-05,2025-06-30,jpm-20250630.htm,0000019617-25-000615


^ 'primaryDocument' is the actual HTML file name we need to download.


### 1b — Download the actual filing (this is HTML)

Our `EdgarClient` (in `ingestion/edgar_client.py`) wraps the two lookups above
plus the download, and adds three production niceties:
- the required **User-Agent** header,
- a **rate limit** (~8 requests/sec, under the SEC's cap of 10),
- **caching** — it saves the file to `data/raw/` so re-running is instant.

Let's download JPMorgan's latest 10-K and look at what actually lands on disk.

In [6]:
from ingestion.edgar_client import EdgarClient

edgar = EdgarClient()
filing = edgar.list_filings("JPM", forms=("10-K",), since="2024-01-01")[0]
raw_path = edgar.download_filing(filing)   # cached — instant if already present

print("Downloaded to :", raw_path)
print("Filing type   :", filing.form, "for fiscal year ending", filing.report_date)
print("File size     :", f"{raw_path.stat().st_size/1e6:.1f} MB")

# Alongside it we save a small 'sidecar' JSON remembering what this file IS,
# so later steps know the ticker/form/period without re-parsing.
print("\nSidecar metadata we saved next to it:")
print(raw_path.with_suffix(".json").read_text())

Downloaded to : C:\Users\subha\OneDrive\Desktop\Subham\finsight\data\raw\JPM\10-K_2025-12-31.htm
Filing type   : 10-K for fiscal year ending 2025-12-31
File size     : 12.9 MB

Sidecar metadata we saved next to it:
{
  "ticker": "JPM",
  "cik": 19617,
  "form": "10-K",
  "accession": "0001628280-26-008131",
  "filing_date": "2026-02-13",
  "report_date": "2025-12-31",
  "primary_document": "jpm-20251231.htm"
}


### 1c — What does the raw data actually look like?

This is the single most important "screenshot" in the whole notebook. **This is
what a modern SEC filing looks like on disk** — it is machine-generated
inline-XBRL HTML. Notice:
- it opens with an XML declaration and a "Created with the Workiva Platform"
  comment (filing software),
- the real sentences are buried inside deeply nested `<div>`/`<span>` tags with
  styling attributes everywhere.

**Nobody — no model, no human — can work with this directly.** That's why the
next steps exist.

In [7]:
raw_html = raw_path.read_bytes()

print("========== RAW FILING: first 900 bytes exactly as downloaded ==========\n")
print(raw_html[:900].decode("utf-8", "replace"))
print("\n... (this continues for", f"{len(raw_html)/1e6:.1f}", "MB) ...")

========== RAW FILING: first 900 bytes exactly as downloaded ==========

<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2026 Workiva-->
<!--r:c7652ff4-626c-4654-b145-19dfe5912602,g:d35e7381-7855-4836-bd1e-2a303b8cc81d,d:1e82dc5e49024170976eb7ddc7c0a10b-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:us-gaap="http://fasb.org/us-gaap/2025" xmlns:srt="http://fasb.org/srt/2025" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:stpr="http://xbrl.sec.gov/stpr/2025" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ecd="http:

... (this continues for 1

In [8]:
# Let's quantify how much of that file is markup vs actual readable text.
from bs4 import BeautifulSoup
soup_full = BeautifulSoup(raw_html, "lxml")
visible = soup_full.get_text(" ", strip=True)
print(f"Raw file (HTML markup + text): {len(raw_html):>12,} characters")
print(f"Just the visible text        : {len(visible):>12,} characters")
print(f"So ~{100*(1-len(visible)/len(raw_html)):.0f}% of the file is markup/tags we must strip away.")

C:\Users\subha\AppData\Local\Temp\ipykernel_22672\3473915437.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup_full = BeautifulSoup(raw_html, "lxml")


Raw file (HTML markup + text):   12,927,325 characters
Just the visible text        :    1,410,533 characters
So ~89% of the file is markup/tags we must strip away.


# Step 2 — Preprocessing: turn messy HTML into clean text

This is the job of `ingestion/parser.py`, function `html_to_text()`. It does
three things. Let's see each on a **tiny toy example** first (so you can see the
mechanism clearly), then on the **real filing**.

The three operations:
1. **Delete `<script>` and `<style>`** — code and styling, never content.
2. **Flatten every table** into pipe-separated rows: `label | 2025 | 2024`.
   This is subtle and important — see below.
3. **Collapse whitespace** but keep line breaks (we need lines to find headings).

### Why flatten tables into `a | b | c` rows?

Financial filings are *full* of tables. If you let an HTML parser rip the text
out naively, a row like:

| Provision for credit losses | 9,320 | 6,899 |

becomes the text `Provision for credit losses 9320 6899` — and now the number
`9,320` has floated away from its label. By joining cells with ` | ` we keep the
number glued to what it means. Watch it happen 👇

In [9]:
from ingestion.parser import html_to_text

toy_html = '''<html><body>
  <style>.hidden{display:none}</style>
  <script>trackAnalytics();</script>
  <div><span style="font-weight:bold">Item 1A.</span> Risk Factors.</div>
  <p>The Firm faces <span>many</span> risks, including&#160;credit risk.</p>
  <table>
    <tr><th>Metric</th><th>2025</th><th>2024</th></tr>
    <tr><td>Provision for credit losses</td><td>9,320</td><td>6,899</td></tr>
  </table>
</body></html>'''

before_after(toy_html, html_to_text(toy_html),
             "BEFORE — raw HTML (tags, script, style, table)",
             "AFTER — html_to_text() output")

See what happened:
- the `<script>` and `<style>` are **gone**,
- the bold/span tags are **gone** but their text survived,
- `&#160;` (a non-breaking space) became a normal space,
- the **table became one line**: `Metric | 2025 | 2024` then
  `Provision for credit losses | 9,320 | 6,899` — number and label stay together.

Now the same function on a **real table from JPMorgan's actual filing** (not a
toy). We'll reach into the real HTML, grab a genuine table, and show it raw vs
flattened.

In [10]:
# Pull a real, small-ish table out of the actual filing to demonstrate.
real_table = None
for t in soup_full.find_all("table"):
    txt = t.get_text(" ", strip=True)
    if 60 < len(txt) < 400 and any(ch.isdigit() for ch in txt) and len(t.find_all("tr")) <= 6:
        real_table = t
        break

raw_table_html = real_table.prettify()[:1500]
flattened = html_to_text(str(real_table))
before_after(raw_table_html, flattened,
             "BEFORE — a real <table> from JPM's 10-K (raw HTML)",
             "AFTER — flattened to pipe-separated rows")

### The whole filing, cleaned

Now we run `html_to_text()` on the entire 12.9 MB file. This takes ~10–20
seconds because the HTML parser has to walk the whole document tree.

In [11]:
%%time
clean_text = html_to_text(raw_html)
print(f"HTML in : {len(raw_html):>12,} chars")
print(f"Text out: {len(clean_text):>12,} chars  (the markup is gone)")

HTML in :   12,927,325 chars
Text out:    1,449,219 chars  (the markup is gone)
CPU times: total: 2.19 s
Wall time: 2.2 s


In [12]:
# Find where the Risk Factors section begins in the cleaned text and show it.
i = clean_text.find("Item 1A. Risk Factors.")
print("========== CLEAN TEXT around 'Item 1A. Risk Factors' ==========\n")
print(clean_text[i:i+800])

========== CLEAN TEXT around 'Item 1A. Risk Factors' ==========

Item 1A. Risk Factors. 
The following discussion sets forth the material risk factors that could affect JPMorganChase’s financial condition and operations. Readers should not consider any descriptions of these factors to be a complete set of all potential risks that could affect the Firm. Any of the risk factors discussed below could by itself, or combined with other factors, materially and adversely affect JPMorganChase’s business, results of operations, financial condition, capital position, liquidity, competitive position or reputation, including by materially increasing expenses or decreasing revenues, which could result in material losses or a decrease in earnings.
Summary
The principal risk factors include:
•
Legal and Regulatory 
risks, including the impact of extensive supervision 


> **A gotcha we hit for real** (worth knowing): on Windows, printing this text
> in a plain terminal sometimes shows `�` where curly apostrophes should be.
> That is **not** corrupted data — the bytes are the correct Unicode character
> `’` (U+2019); it's just the Windows console's default code page failing to
> display it. Running Python with `-X utf8` fixes the display. Inside this
> notebook it renders fine.

# Step 3 — Split the clean text into named sections

We now have one giant blob of clean text. But a 10-K has structure — those
numbered **Items**. We want to split the blob at each "Item N." heading so that
later, when someone asks about *risks*, we can search **only** the Risk Factors
section (Item 1A) instead of the whole document.

We find headings with a **regular expression** that matches lines like
`Item 1A.` or `ITEM 7:`. But there's a classic trap.

### The table-of-contents trap

Every 10-K has a table of contents at the top that *also* lists "Item 1A. Risk
Factors", "Item 7. MD&A", etc. So each item name appears **at least twice** in
the document: once in the TOC (just the title, a few characters) and once as the
real section (thousands of characters of body text).

The fix in `split_items()`: after each heading, measure how much text follows
before the next heading. If it's **less than 200 characters**, it's a TOC entry
→ discard it. Keep the substantial one. Let's watch it work.

In [13]:
from ingestion.parser import _ITEM_RE, split_items

# Every place the regex thinks it found an "Item N" heading:
matches = list(_ITEM_RE.finditer(clean_text))
print("The regex pattern:", _ITEM_RE.pattern)
print(f"\nIt found {len(matches)} raw 'Item ...' matches in the document.")
print("\nThe FIRST 12 matches (notice the tight cluster near the top — that's the")
print("table of contents, all crammed together with little text between them):\n")
for m in matches[:12]:
    following = clean_text[m.start():m.start()+400]
    gap = "TOC-like (short)" if len(clean_text[m.start(): (matches[matches.index(m)+1].start() if matches.index(m)+1 < len(matches) else len(clean_text))]) < 200 else "real section"
    print(f"  char {m.start():>9,}  |  Item {m.group(1):<3}  |  {gap}")

The regex pattern: (?:^|\n)\s*ITEM\s+(\d{1,2}[A-C]?)\s*[.:—\-]

It found 43 raw 'Item ...' matches in the document.

The FIRST 12 matches (notice the tight cluster near the top — that's the
table of contents, all crammed together with little text between them):

  char   238,695  |  Item 1    |  real section
  char   238,950  |  Item 1A   |  TOC-like (short)
  char   238,982  |  Item 1B   |  TOC-like (short)
  char   239,059  |  Item 2    |  TOC-like (short)
  char   239,086  |  Item 3    |  TOC-like (short)
  char   239,120  |  Item 4    |  TOC-like (short)
  char   239,168  |  Item 5    |  TOC-like (short)
  char   239,293  |  Item 6    |  TOC-like (short)
  char   239,317  |  Item 7    |  TOC-like (short)
  char   239,419  |  Item 7A   |  TOC-like (short)
  char   239,495  |  Item 8    |  TOC-like (short)
  char   239,555  |  Item 9    |  TOC-like (short)


In [14]:
# Now the cleaned result: TOC duplicates dropped, one row per real section.
sections = split_items(clean_text, "10-K")

sections_df = pd.DataFrame([
    {"item": s.item, "section_name": s.name[:55], "chars": len(s.text),
     "≈ pages": round(len(s.text)/3000)}
    for s in sections
])
print(f"{len(matches)} raw matches  ->  {len(sections)} real sections after dropping TOC duplicates\n")
display(sections_df)

43 raw matches  ->  12 real sections after dropping TOC duplicates



,item,section_name,chars,≈ pages
0,1,Business,39406,13
1,1A,Risk Factors,112633,38
2,2,Properties,2148,1
3,5,Market for Registrant's Common Equity,2747,1
4,7,Management's Discussion and Analysis (MD&A),395,0
5,7A,Quantitative and Qualitative Disclosures About Market R,270,0
6,8,Financial Statements and Supplementary Data,370,0
7,9A,Controls and Procedures,2229,1
8,9B,Other Information,5493,2
9,10,"Directors, Executive Officers and Corporate Governance",3996,1


**Two real-world observations** in that table (both documented in the repo):

- **Item 7 (MD&A) looks tiny** and **Item 15 is enormous (~1M chars).** That's
  because JPMorgan legally "incorporates by reference" — the actual MD&A and
  financial statements physically live inside the Item 15 exhibit blob. The
  content is all there and gets indexed; it's just labelled Item 15. A real
  data-quality quirk you'd note and fix later, not hide.
- This is exactly the kind of messy-real-data surprise that makes retrieval
  quality (Phase 2) a measured, iterative process rather than a one-shot.

In [15]:
# Let's actually LOOK at a real section's text — the start of Risk Factors.
risk = next(s for s in sections if s.item == "1A")
print(f"===== Item {risk.item}: {risk.name}  ({len(risk.text):,} chars) =====\n")
print(risk.text[:1000])

===== Item 1A: Risk Factors  (112,633 chars) =====

Item 1A. Risk Factors. 
The following discussion sets forth the material risk factors that could affect JPMorganChase’s financial condition and operations. Readers should not consider any descriptions of these factors to be a complete set of all potential risks that could affect the Firm. Any of the risk factors discussed below could by itself, or combined with other factors, materially and adversely affect JPMorganChase’s business, results of operations, financial condition, capital position, liquidity, competitive position or reputation, including by materially increasing expenses or decreasing revenues, which could result in material losses or a decrease in earnings.
Summary
The principal risk factors include:
•
Legal and Regulatory 
risks, including the impact of extensive supervision and regulation, as well as changes to or in the application, interpretation or enforcement of applicable law or executive branch actions, on JPMorga

# Step 4 — Chunking: cut sections into bite-sized passages

### Why chunk at all? (two reasons)

1. **Embedding models have an input limit.** You can't feed a 112,000-character
   Risk Factors section into the embedder as one vector — and even if you could,
   compressing 40 pages into a single vector would blur everything together.
2. **Retrieval works best on focused passages.** When someone asks about
   ransomware, we want to return *the paragraph about ransomware*, not the whole
   40-page section.

**CV analogy:** you don't feed a 10,000×10,000 image into a network as one
tensor — you tile it into patches. Chunking is tiling, for text.

### How we chunk (the design decisions)

- **Split on paragraph boundaries**, targeting ~4,000 characters per chunk.
  Never cut mid-sentence.
- **One-paragraph overlap** between consecutive chunks: the last paragraph of
  chunk *N* is repeated as the first paragraph of chunk *N+1*. Why? So a fact
  that sits right on a boundary isn't split away from its context. (Like
  overlapping image tiles so an object on the seam isn't cut in half.)
- **Prepend a metadata header** to every chunk:
  `[JPM | 10-K FY2025 | Item 1A: Risk Factors]`. This is a small but powerful
  trick — it means the *embedding* captures the provenance, and the *LLM* later
  sees which company/section a passage came from.
- **Attach a ready-made citation** string to every chunk:
  `[JPM 10-K 2025, Item 1A]`. This travels with the chunk all the way to the
  final answer, which is how every claim ends up cited.

Let's chunk the Risk Factors section and watch it happen.

In [16]:
from ingestion.chunker import chunk_section, _split_paragraphs

# First, how many paragraphs does the section break into?
paras = _split_paragraphs(risk.text)
print(f"Item 1A is {len(risk.text):,} chars = {len(paras)} paragraphs.")
print(f"\nParagraph lengths (first 15): {[len(p) for p in paras[:15]]}")
print("\n--- Paragraph #2 (a real risk paragraph) ---")
print(textwrap.fill(paras[2][:600], 95))

Item 1A is 112,633 chars = 24 paragraphs.

Paragraph lengths (first 15): [5240, 4959, 5239, 5089, 5307, 4165, 5160, 5193, 5504, 5163, 4598, 4719, 5032, 4764, 4665]

--- Paragraph #2 (a real risk paragraph) ---
that are stricter than a global standard, which could create competitive disadvantages for
those firms, such as JPMorganChase, that are subject to the enhanced regulations. Furthermore,
certain authorities outside the U.S. have adopted applicable law that could conflict with or
prohibit JPMorganChase from complying with applicable law in other jurisdictions, which could
create conflict of law issues and could increase risks associated with non-compliance.
Regulatory initiatives outside the U.S. have required and could in the future require
JPMorganChase to significantly modify its operations o


In [17]:
# Now group those paragraphs into ~4000-char chunks with 1-paragraph overlap.
chunks = chunk_section(risk, ticker="JPM", form="10-K", period=filing.report_date)

print(f"{len(paras)} paragraphs  ->  {len(chunks)} chunks\n")
chunk_df = pd.DataFrame([
    {"seq": c.seq, "chars": len(c.text), "citation": c.citation,
     "starts_with": c.text.split(chr(10))[0][:45]}
    for c in chunks
])
display(chunk_df.head(8))
print("Every chunk starts with that [ ... ] metadata header line. ^")

24 paragraphs  ->  24 chunks



,seq,chars,citation,starts_with
0,0,5284,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
1,1,10245,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
2,2,10244,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
3,3,10374,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
4,4,10442,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
5,5,9518,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
6,6,9371,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]
7,7,10399,"[JPM 10-K 2025, Item 1A]",[JPM | 10-K FY2025 | Item 1A: Risk Factors]


Every chunk starts with that [ ... ] metadata header line. ^


### Look at one complete chunk, exactly as it will be embedded and stored

Notice the very first line is the metadata header we injected. Everything after
it is real filing text.

In [18]:
print("========== CHUNK #3 (verbatim) ==========\n")
print(chunks[3].text[:1400])
print("\n... [truncated] ...")
print("\n--- and its attached citation ---")
print(chunks[3].citation)

========== CHUNK #3 (verbatim) ==========

[JPM | 10-K FY2025 | Item 1A: Risk Factors]
that are stricter than a global standard, which could create competitive disadvantages for those firms, such as JPMorganChase, that are subject to the enhanced regulations. Furthermore, certain authorities outside the U.S. have adopted applicable law that could conflict with or prohibit JPMorganChase from complying with applicable law in other jurisdictions, which could create conflict of law issues and could increase risks associated with non-compliance.
Regulatory initiatives outside the U.S. have required and could in the future require JPMorganChase to significantly modify its operations or legal entity structure in the places in which those initiatives are implemented, such as requirements for: 
•
establishing locally-based intermediate holding companies or operating subsidiaries
•
maintaining minimum amounts of capital or liquidity in locally-based subsidiaries
•
implementing processes within l

### Proof of the overlap

Let's confirm the "one-paragraph overlap" claim: the **last** paragraph of
chunk 3 should be **identical** to the **first body paragraph** of chunk 4.

In [19]:
tail_of_3 = chunks[3].text.split("\n\n")[-1]
# chunk 4: line 0 is the metadata header, so the first body paragraph is next
body_of_4 = chunks[4].text.split("\n", 1)[1].split("\n\n")[0]

before_after(tail_of_3[:500], body_of_4[:500],
             "LAST paragraph of chunk #3", "FIRST body paragraph of chunk #4")
print("They match => the overlap is working. A fact on the boundary lives in BOTH chunks.")

They match => the overlap is working. A fact on the boundary lives in BOTH chunks.


# Step 5 — How the `.jsonl` files are formed and what they hold

Now we can answer your exact question: *"how do `JPM_10-K_2025-12-31.jsonl` and
the others get formed?"*

After chunking, each `Chunk` object is just a bundle of fields (ticker, form,
period, item, seq, text, citation…). We write **one chunk per line** as a JSON
object into a file named `<TICKER>_<FORM>_<PERIOD>.jsonl`. That's what
`ingestion/pipeline.py` does for every filing.

### Why `.jsonl` (JSON Lines) and not one big `.json`?

- A `.jsonl` file is **one independent JSON object per line**. You can read it
  line by line without loading the whole file into memory, append to it, and
  eyeball individual records. It's the standard format for ML datasets (you've
  probably seen it for training data).
- It's the **hand-off point** between the "download + process" stage (free, no
  API key) and the "embed + store" stage (needs the Azure key). We can re-run
  parsing cheaply without touching the database or paying for embeddings.

Let's look at the **real file on disk**.

In [20]:
from config import DATA_PARSED

jsonl_path = DATA_PARSED / "JPM_10-K_2025-12-31.jsonl"
lines = jsonl_path.read_text(encoding="utf-8").splitlines()
print("File:", jsonl_path.name)
print("Number of lines (= number of chunks):", len(lines))
print(f"File size: {jsonl_path.stat().st_size/1e6:.2f} MB")

print("\n========== The RAW first line, exactly as stored on disk ==========\n")
print(lines[0][:700], "...")

File: JPM_10-K_2025-12-31.jsonl
Number of lines (= number of chunks): 475
File size: 2.37 MB

========== The RAW first line, exactly as stored on disk ==========

{"ticker": "JPM", "form": "10-K", "period": "2025-12-31", "fiscal_label": "FY2025", "item": "1", "section_name": "Business", "seq": 0, "text": "[JPM | 10-K FY2025 | Item 1: Business]\nItem 1. Business.\nOverview\nJPMorgan Chase & Co. (\u201cJPMorganChase\u201d or the \u201cFirm\u201d, NYSE: JPM), a financial holding company incorporated under Delaware law in 1968, is a leading financial services firm based in the United States of America (\u201cU.S.\u201d), with operations worldwide. JPMorganChase had $4.4 trillion in assets and $362.4 billion in stockholders\u2019 equity as of December 31, 2025. The Firm is a leader in investment banking, financial services for consumers and small businesse ...


In [21]:
# Each line is a JSON object. Parse one and show its fields as a table.
record = json.loads(lines[0])
field_table = pd.DataFrame(
    [(k, (v[:80] + "…") if isinstance(v, str) and len(v) > 80 else v) for k, v in record.items()],
    columns=["field", "value"]
)
display(field_table)

,field,value
0,ticker,JPM
1,form,10-K
2,period,2025-12-31
3,fiscal_label,FY2025
4,item,1
5,section_name,Business
6,seq,0
7,text,[JPM | 10-K FY2025 | Item 1: Business]\nItem 1. Business.\nOverview\nJPMorgan Chase…
8,citation,"[JPM 10-K 2025, Item 1]"


In [22]:
# And here is the code (from ingestion/pipeline.py) that produced these files,
# so you see there is no magic — chunk objects -> dict -> json line.
import dataclasses
example_line = json.dumps(dataclasses.asdict(chunks[0]))
print("A Chunk object serialized to one JSONL line looks exactly like:\n")
print(example_line[:400], "...")

A Chunk object serialized to one JSONL line looks exactly like:

{"ticker": "JPM", "form": "10-K", "period": "2025-12-31", "fiscal_label": "FY2025", "item": "1A", "section_name": "Risk Factors", "seq": 0, "text": "[JPM | 10-K FY2025 | Item 1A: Risk Factors]\nItem 1A. Risk Factors. \nThe following discussion sets forth the material risk factors that could affect JPMorganChase\u2019s financial condition and operations. Readers should not consider any descriptions ...


# Step 6 — Embeddings: turning text into vectors

This is the concept most familiar to you from CV, just applied to text.

**An embedding is a list of numbers (a vector) that captures the *meaning* of a
piece of text.** A model called `text-embedding-3-large` reads a chunk and
outputs **3072 floating-point numbers**. The key property:

> Texts with **similar meaning** get **similar vectors** — even if they share no
> words. "loan loss provision" and "money set aside for bad loans" land close
> together; "rooftop garden" lands far away.

This is *exactly* like a CNN mapping images to feature vectors where similar
images are close. Same idea, different modality.

Let's make it concrete: embed three sentences and measure their distances.

In [23]:
from ingestion.embedder import get_openai, embed_texts

aoai = get_openai()   # connects to Azure OpenAI using your .env credentials

sentences = [
    "The provision for credit losses increased due to loan deterioration.",   # A
    "Reserves set aside for loans that may not be repaid grew this quarter.",  # B: same meaning as A, different words
    "Our new headquarters features a rooftop garden for employees.",          # C: unrelated
]
vectors = embed_texts(aoai, sentences)

print("Each sentence became a vector of length:", len(vectors[0]))
print("\nFirst 8 numbers of sentence A's 3072-dim vector:")
print([round(x, 4) for x in vectors[0][:8]], "...")

Each sentence became a vector of length: 3072

First 8 numbers of sentence A's 3072-dim vector:
[-0.0453, 0.0071, -0.0026, 0.0245, 0.0061, -0.0208, -0.0154, 0.0303] ...


In [24]:
import math
def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b)))

print("Cosine similarity (1.0 = identical meaning, 0 = unrelated):\n")
print(f"  A vs B (both about credit losses): {cosine(vectors[0], vectors[1]):.3f}   <- HIGH, as expected")
print(f"  A vs C (credit losses vs garden) : {cosine(vectors[0], vectors[2]):.3f}   <- LOW, as expected")
print(f"  B vs C (bad loans   vs garden)   : {cosine(vectors[1], vectors[2]):.3f}   <- LOW")
print("\nThe model understood MEANING, not keywords: A and B share almost no words")
print("yet score high. This is what makes 'search by meaning' possible.")

Cosine similarity (1.0 = identical meaning, 0 = unrelated):

  A vs B (both about credit losses): 0.604   <- HIGH, as expected
  A vs C (credit losses vs garden) : 0.071   <- LOW, as expected
  B vs C (bad loans   vs garden)   : 0.132   <- LOW

The model understood MEANING, not keywords: A and B share almost no words
yet score high. This is what makes 'search by meaning' possible.


# Step 7 — Vector database vs. a traditional database

Now the question you asked directly: **how is a vector DB different from a normal
database, and how did we set ours up?**

### The core difference

| | **Traditional database** (what you know) | **Vector database** (what RAG needs) |
|---|---|---|
| Stores | rows of numbers, dates, strings | rows **plus** high-dim vectors |
| You ask | "rows WHERE ticker = 'JPM'" (exact match) | "the 8 rows whose vector is **nearest** to *this* vector" |
| Question type | exact / filtering | **similarity** / nearest-neighbour |
| CV analogy | a SQL table | a **FAISS** index |

A traditional DB answers *"find rows equal to X."* A vector DB answers *"find
rows **most similar in meaning** to X"* — which is fuzzy, ranked, and exactly
what "find the most relevant paragraph" needs.

### The good news: we don't need a separate exotic system

We use **PostgreSQL** (a rock-solid traditional database) plus an extension
called **pgvector** that adds a `vector` column type and nearest-neighbour
search. So one database does **both** jobs:
- exact filters (`WHERE ticker = 'JPM' AND item = '1A'`) — traditional DB
- similarity search (`ORDER BY embedding <=> query_vector`) — pgvector

That `<=>` is pgvector's "distance between vectors" operator. More on it in
Step 8.

### How the database was set up (no manual steps)

It's all declared in `infra/`:
- `docker-compose.yml` runs the image `pgvector/pgvector:pg16` — Postgres 16
  with pgvector pre-installed — as a container named `finsight-db`.
- It mounts `schema.sql` into a special folder (`/docker-entrypoint-initdb.d/`).
  Postgres automatically runs any SQL there **the first time it starts** on an
  empty data volume. That's why the table exists without us ever running a
  migration by hand.

Let's connect to the **live database** and inspect it.

In [25]:
import psycopg
from config import DATABASE_URL

conn = psycopg.connect(DATABASE_URL)
print("Connected to the live Postgres database.\n")

# Show the actual schema of the 'chunks' table, column by column.
schema = conn.execute("""
    SELECT column_name, data_type, character_maximum_length
    FROM information_schema.columns
    WHERE table_name = 'chunks'
    ORDER BY ordinal_position
""").fetchall()
schema_df = pd.DataFrame(schema, columns=["column", "type", "max_len"])
display(schema_df)
print("Note the 'embedding' column: type USER-DEFINED = pgvector's vector(3072).")
print("Note the 'tsv' column: a tsvector for keyword search, pre-built for Phase 2.")

Connected to the live Postgres database.



,column,type,max_len
0,id,bigint,None
1,ticker,text,None
2,form,text,None
3,period,date,None
4,fiscal_label,text,None
5,item,text,None
6,section_name,text,None
7,seq,integer,None
8,citation,text,None
9,text,text,None


Note the 'embedding' column: type USER-DEFINED = pgvector's vector(3072).
Note the 'tsv' column: a tsvector for keyword search, pre-built for Phase 2.


### What each column is for

| Column | Purpose |
|---|---|
| `id` | auto-increment primary key (just a row number) |
| `ticker, form, period, fiscal_label, item, section_name, seq` | the **metadata** — used for exact SQL filtering and to rebuild context |
| `citation` | the ready-made source label, e.g. `[JPM 10-K 2025, Item 1A]` |
| `text` | the chunk content (with its metadata header line) |
| `embedding vector(3072)` | the **meaning vector** — this is the "vector DB" part |
| `tsv tsvector` | a keyword-search index, pre-staged for Phase 2's hybrid search |

There's also a `UNIQUE (ticker, form, period, item, seq)` constraint. That makes
loading **idempotent**: if you re-run the loader, duplicate chunks are ignored
instead of inserted twice. Safe to re-run after a crash.

### Real rows from the live table

Here are actual stored rows (metadata columns only — the vector is 3072 numbers,
shown separately next).

In [26]:
rows = conn.execute("""
    SELECT id, ticker, form, period, fiscal_label, item, seq, citation
    FROM chunks
    WHERE ticker = 'JPM' AND item = '1A'
    ORDER BY seq
    LIMIT 5
""").fetchall()
display(pd.DataFrame(rows, columns=["id","ticker","form","period","fiscal_label","item","seq","citation"]))

# How many rows total, per company?
counts = conn.execute("SELECT ticker, count(*) FROM chunks GROUP BY ticker ORDER BY ticker").fetchall()
print("Total chunks stored, per company:")
for tkr, n in counts:
    print(f"   {tkr}: {n}")

,id,ticker,form,period,fiscal_label,item,seq,citation
0,266,JPM,10-K,2025-12-31,FY2025,1A,0,"[JPM 10-K 2025, Item 1A]"
1,267,JPM,10-K,2025-12-31,FY2025,1A,1,"[JPM 10-K 2025, Item 1A]"
2,268,JPM,10-K,2025-12-31,FY2025,1A,2,"[JPM 10-K 2025, Item 1A]"
3,269,JPM,10-K,2025-12-31,FY2025,1A,3,"[JPM 10-K 2025, Item 1A]"
4,270,JPM,10-K,2025-12-31,FY2025,1A,4,"[JPM 10-K 2025, Item 1A]"


Total chunks stored, per company:
   BAC: 256
   JPM: 475


In [27]:
# What does the 'embedding' column actually contain? A vector of 3072 floats.
# (We only print the first few — the full thing is one row of 3072 numbers.)
emb_preview = conn.execute(
    "SELECT left(embedding::text, 90) FROM chunks WHERE ticker='JPM' AND item='1A' AND seq=0"
).fetchone()[0]
print("The 'embedding' cell for one chunk starts like:")
print(emb_preview, "...]  (3072 numbers total)")

print("\nSo each row = human-readable metadata + text + a 3072-dim meaning vector,")
print("all living together in ONE table. That's the whole trick.")

The 'embedding' cell for one chunk starts like:
[-0.011184692,-0.042633057,-0.007297516,0.036834717,0.008453369,-0.03390503,-0.0063209534, ...]  (3072 numbers total)

So each row = human-readable metadata + text + a 3072-dim meaning vector,
all living together in ONE table. That's the whole trick.


### How does a chunk get *into* the table? (the storage step)

`ingestion/embedder.py` ties Steps 5–7 together in a loop, per filing:

1. read the `.jsonl` file (the chunks from Step 5),
2. send batches of 64 chunk texts to the embedding model → get vectors (Step 6),
3. `INSERT` each chunk's metadata + text + vector as one row (this step).

The SQL is essentially:

```sql
INSERT INTO chunks (ticker, form, period, item, seq, citation, text, embedding)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (ticker, form, period, item, seq) DO NOTHING;
```

The `ON CONFLICT ... DO NOTHING` is the idempotency guard. You run
`python -m ingestion.embedder` once; it takes ~10 min on the free Azure tier
(rate-limited), and afterwards all 731 rows are in the table — which is the
state we're querying live right now.

# Step 8 — Retrieval and the final cited answer

Everything so far was **preparation** (build the open book). Now the **query
time** flow, which happens every time a user asks something:

1. Embed the **question** with the *same* model → a 3072-dim query vector.
2. Ask pgvector for the chunks whose vectors are **nearest** to it
   (`ORDER BY embedding <=> query_vector`), optionally filtered by company.
3. Paste those chunks into a prompt and let the LLM write the answer — **using
   only that evidence, citing every claim, or refusing.**

`<=>` is pgvector's cosine-distance operator: smaller distance = more similar.
We convert it to a 0–1 similarity score for readability. Let's retrieve.

In [28]:
from retrieval.search import search

question = "How exposed is the bank to cyber attacks and ransomware?"
hits = search(question, k=5, tickers=["JPM"])

print(f"Question: {question}\n")
display(pd.DataFrame([
    {"score": round(h.score, 3), "citation": h.citation, "section": h.section_name[:35],
     "text_preview": h.text[len(h.citation)+30:150].replace(chr(10), " ")}
    for h in hits
]))
print("score ~0.6–0.75 = strong semantic match. These are the paragraphs the")
print("LLM will be allowed to use — nothing else.")

Question: How exposed is the bank to cyber attacks and ransomware?



,score,citation,section,text_preview
0,0.561,"[JPM 10-K 2025, Item 1A]",Risk Factors,"ade by JPMorganChase or another market participant, whether inadvertent or malicious, ..."
1,0.538,"[JPM 10-K 2025, Item 15]",Exhibits and Financial Statement Sc,"tatement Schedules (incl. content incorporated by reference: MD&A, financial statement..."
2,0.524,"[JPM 10-K 2025, Item 1A]",Risk Factors,"ctions by banking regulators, as well as changes in applicable law or how applicable l..."
3,0.507,"[JPM 10-K 2025, Item 1A]",Risk Factors,"organChase or its clients, customers, counterparties or employees • manipulate or dest..."
4,0.493,"[JPM 10-K 2025, Item 15]",Exhibits and Financial Statement Sc,"tatement Schedules (incl. content incorporated by reference: MD&A, financial statement..."


score ~0.6–0.75 = strong semantic match. These are the paragraphs the
LLM will be allowed to use — nothing else.


### The final step: the LLM writes a cited answer

The retrieved chunks are pasted into a prompt with strict rules (the "system
prompt" from `rag_cli.py`). The rules, in priority order:
1. use **only** the provided evidence,
2. **cite every claim** with its bracketed source,
3. if the evidence doesn't answer it, say **"insufficient evidence"** — never guess.

In [29]:
from rag_cli import SYSTEM_PROMPT
from config import CHAT_DEPLOYMENT

def rag_answer(question, tickers=None, k=8):
    hits = search(question, k=k, tickers=tickers)
    evidence = "\n\n---\n\n".join(f"PASSAGE {i+1} {h.citation}:\n{h.text}"
                                  for i, h in enumerate(hits))
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": f"EVIDENCE:\n\n{evidence}\n\nQUESTION: {question}"}],
    )
    return resp.choices[0].message.content

ans = rag_answer("What risks did JPMorgan flag about artificial intelligence?", tickers=["JPM"])
display(Markdown(ans))

JPMorgan flagged the following AI-related risks in its risk factors [JPM 10-K 2025, Item 1A]:

- Failures, inappropriate use, lack of transparency, or inaccurate/biased outputs from AI (including generative AI) that could disrupt operations, cause erroneous transactions, compromise data privacy, infringe IP, harm clients, or impair business decisions [JPM 10-K 2025, Item 1A].  
- Increased exposure to cyber attacks, system manipulation, or data loss if AI systems (particularly agentic systems) are not designed with safeguards to prevent unauthorized access or actions [JPM 10-K 2025, Item 1A].  
- Intensified AI-enabled cyber threats (e.g., advanced social engineering, reverse‑engineering of security patches) that could lead to unauthorized access and data breaches [JPM 10-K 2025, Item 1A].  
- Regulatory and compliance challenges from rapidly evolving laws and inconsistent international standards that could raise costs, lead to fines/sanctions, and restrict JPMorganChase’s AI use [JPM 10-K 2025, Item 1A].  
- Competitive disadvantage if competitors deploy AI more quickly or effectively, leading to lost market share, reduced cost efficiency, or weaker client experience [JPM 10-K 2025, Item 1A].  
- Replacement or disintermediation of direct customer relationships if AI agents autonomously manage or intermediate customer financial decisions and product selection [JPM 10-K 2025, Item 1A].  
- Miscalibrated workforce planning or over‑reliance on AI that could cause staff shortages, hinder skill development (critical thinking, judgment, creativity), or otherwise impair workforce capabilities [JPM 10-K 2025, Item 1A].  
- Use of AI by customers or bad actors in unexpected or disruptive ways, or AI systems being breached/infiltrated, which could increase compliance expenses and reduce income from affected products and services [JPM 10-K 2025, Item 1A].

In [30]:
# The trust test: ask something NOT in the filings. A plain chatbot would
# happily invent an answer. FinSight must refuse.
ans = rag_answer("What did Tesla say about vehicle recall risks?", tickers=["JPM", "BAC"])
display(Markdown("**Answer:**\n\n" + ans))

**Answer:**

Insufficient evidence in the indexed filings.

Missing from the provided indexed filings: any Tesla filings or any statements by Tesla regarding vehicle recall risks.

In [31]:
conn.close()
print("Done. Database connection closed.")

Done. Database connection closed.


# Recap — the whole pipeline, in the order you just did it

| Step | What went in | What came out | Key idea |
|---|---|---|---|
| 1 | ticker "JPM" | 12.9 MB raw HTML filing | SEC EDGAR is free; identify yourself |
| 2 | raw HTML | clean text | flatten tables so numbers keep their labels |
| 3 | clean text | ~12 named sections | drop table-of-contents duplicates |
| 4 | one section | ~4k-char chunks | metadata header + citation + overlap on every chunk |
| 5 | chunk objects | `.jsonl` file (one chunk/line) | cheap, re-runnable hand-off before embedding |
| 6 | chunk text | 3072-dim vector | similar meaning → similar vector (like CNN features) |
| 7 | vectors + metadata | rows in Postgres+pgvector | one table does exact filters **and** similarity search |
| 8 | a question | cited answer, or a refusal | retrieve nearest chunks → LLM answers from them only |

### What you now understand that you didn't before
- **RAG = open-book exam**: retrieve first, then answer with citations.
- **Embeddings** are feature vectors for text; **cosine similarity** compares meaning.
- A **vector database** is a traditional DB that can also do nearest-neighbour
  search on those vectors — here, Postgres + pgvector in one Docker container.
- The `.jsonl` files are **our** intermediate output, not SEC input.
- **Chunking with metadata headers + citations** is the design decision that
  makes every final answer traceable to a real source.

### Next: Phase 2 — is our retrieval actually any good?
Right now we *believe* retrieval works because the demos look right. Phase 2
replaces belief with **measurement**: a golden dataset of ~50 Q&A, RAGAS
metrics, then keyword+vector **hybrid search** and a **reranker** — improving
each number deliberately. That's the real craft of RAG.